In [1]:
import os
import json
from graphviz import Digraph
import numpy as np
import matplotlib.colors as mcolors
import random
from plyfile import PlyData
import open3d as o3d
scan_id = '752cc5a3-920c-26f5-8ff3-49518eff94c6'
ply_file_path = f"/mnt/projects/open3dsg/data/3RScan/data/{scan_id}/labels.instances.annotated.v2.ply"
with open('/mnt/projects/open3dsg/data/3RScan/3DSSG_subset/relationships_train.json') as file:
    scans = json.load(file)['scans']
scan_indices = []
for i, scan in enumerate(scans):
    if scan['scan'] == scan_id:
        scan_indices.append(i)
object_id_split = scans[scan_indices[0]]['objects'].keys() # scan_indices[0] is split 1

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
object_id_split = [int(o_id) for o_id in object_id_split]
object_id_split

[1, 18, 84, 85, 17, 63, 61, 62, 15]

In [3]:
ply_data = PlyData.read(ply_file_path)

In [4]:
valid_vertex = np.isin(ply_data['vertex']['objectId'], list(object_id_split))

In [5]:
# vertex of split1
x_array = ply_data['vertex']['x'][valid_vertex]
y_array = ply_data['vertex']['y'][valid_vertex]
z_array = ply_data['vertex']['z'][valid_vertex]
vertex_positions = np.array([x_array, y_array, z_array]).transpose(1, 0)
# color of split 1
red_array = ply_data['vertex']['red'][valid_vertex]
green_array = ply_data['vertex']['green'][valid_vertex]
blue_array = ply_data['vertex']['blue'][valid_vertex]
vertex_colors = np.array([red_array, green_array, blue_array]).transpose(1, 0)
vertex_colors = vertex_colors.astype(np.float64) / 255.0

In [6]:
valid_vertex_indices = np.where(valid_vertex)[0]
mapping = {old_idx: new_idx for new_idx, old_idx in enumerate(valid_vertex_indices)}
face_vertices = ply_data['face']['vertex_indices']
face_split = []
for face_vertex in face_vertices:
    if face_vertex[0] in valid_vertex_indices and face_vertex[1] in valid_vertex_indices and face_vertex[2] in valid_vertex_indices:
        face_split.append(np.array([mapping[face_vertex[0]], mapping[face_vertex[1]], mapping[face_vertex[2]]]))
mesh = o3d.geometry.TriangleMesh()
mesh.vertices = o3d.utility.Vector3dVector(vertex_positions)
mesh.vertex_colors = o3d.utility.Vector3dVector(vertex_colors)
mesh.triangles = o3d.utility.Vector3iVector(face_split)
os.makedirs('mesh_split', exist_ok=True)
o3d.io.write_triangle_mesh(f"mesh_split/{scan_id}.ply", mesh, write_ascii=True)

True